# register-back-fn-after-wrap — worked example 3: Register Backward Functions for Addition at Both Argument Positions

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-back-fn-after-wrap`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For a binary operation like `add(x, y) = x + y`, both `x` and `y` are differentiable inputs. You must register a separate backward function for each argument position: argnum=0 for the gradient with respect to `x`, and argnum=1 for the gradient with respect to `y`. For addition, both partial derivatives equal 1, so both back functions just pass `grad_out` through unchanged.

## Worked solution

**Step 1 — math.** `out = x + y`. So `d(out)/d(x) = 1` and `d(out)/d(y) = 1`. Both backward functions return `grad_out * 1 = grad_out`.

**Step 2 — define back functions.** `add_back0(grad_out, out, x, y)` returns `grad_out`. `add_back1(grad_out, out, x, y)` also returns `grad_out`. They accept all forward args by convention even though they don't use them all.

**Step 3 — register both.** Two calls: `add_back_func(t.add, 0, add_back0)` and `add_back_func(t.add, 1, add_back1)`. Both go into the same table.

**Step 4 — dispatch both paths.** Retrieve each back fn by argnum and call it. Both should return tensors equal to `grad_out`.

**Step 5 — print and verify.** A backprop loop would call both; here we do it manually to see the symmetry.

In [ ]:
import torch as t

class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]

def add_back0(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, y: t.Tensor) -> t.Tensor:
    # d(x+y)/dx = 1
    return grad_out

def add_back1(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor, y: t.Tensor) -> t.Tensor:
    # d(x+y)/dy = 1
    return grad_out

def register_add(BACK_FUNCS: BackwardFuncLookup) -> None:
    BACK_FUNCS.add_back_func(t.add, 0, add_back0)
    BACK_FUNCS.add_back_func(t.add, 1, add_back1)

# --- exercise and print ---
BACK_FUNCS = BackwardFuncLookup()
register_add(BACK_FUNCS)

x = t.tensor([1.0, 2.0, 3.0])
y = t.tensor([4.0, 5.0, 6.0])
out = t.add(x, y)
grad_out = t.tensor([0.5, 1.0, 2.0])

for argnum in [0, 1]:
    back_fn = BACK_FUNCS.get_back_func(t.add, argnum)
    g = back_fn(grad_out, out, x, y)
    print(f'argnum={argnum}: grad = {g}')  # Both should equal grad_out

print('table size:', len(BACK_FUNCS._table))  # 2 entries